# Stress Prediction — 원본 구조 유지 미세 튜닝 버전

이 노트북은 기존 제출 파일의 구조를 **그대로 유지**하면서 다음 네 값만 탐색합니다.

- `TREE_QUANTILE`
- `PAIR_QUANTILE`
- `PAIR_WEIGHT`
- `OVERRIDE_THRESHOLD`

추가 파생변수, 모델 구조 변경, 반올림 제거는 적용하지 않습니다. 먼저 원본 설정의 CV 점수를 재현한 뒤 캐시를 이용해 미세 탐색합니다.

In [1]:
from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
WINSORIZE_COLUMNS = ["mean_working", "diastolic_blood_pressure", "cholesterol", "glucose"]

# 기존 로컬 경로 유지
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 원본 제출 설정
BASE_TREE_QUANTILE = 0.54
BASE_PAIR_QUANTILE = 0.48
BASE_PAIR_WEIGHT = 0.28
BASE_OVERRIDE_THRESHOLD = 0.20
ROUND_DECIMALS = 2

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


In [2]:
def compute_iqr_bounds(train_frame, columns):
    bounds = {}
    for col in columns:
        q1, q3 = train_frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return bounds


def apply_winsorize(frame, bounds):
    result = frame.copy()
    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(lower, upper)
    return result


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    # 원본 파일의 파생변수만 유지
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def nearest_neighbor_lookup(train_raw, val_raw, target, numeric_cols):
    train_numeric = train_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    val_numeric = val_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric)
    val_scaled = scaler.transform(val_numeric)
    tree = cKDTree(train_scaled)
    nearest_dist, nearest_idx = tree.query(val_scaled, k=1)
    nearest_target = target.to_numpy(dtype=float)[nearest_idx]
    return nearest_dist, nearest_target


## 1. Fold별 원시 예측 캐시 생성

ExtraTrees는 Fold마다 한 번만 학습합니다. 이후 분위수·가중치·임계값 조합은 재학습 없이 빠르게 비교합니다.

In [3]:
def tree_prediction_matrix(train_features, val_features, target):
    train_no_gender = train_features.drop(columns=["gender"])
    val_no_gender = val_features.drop(columns=["gender"])
    weighted_train = add_feature_copies(train_no_gender)
    weighted_val = add_feature_copies(val_no_gender)

    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)

    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        criterion="squared_error",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(train_matrix, target)

    return np.column_stack([
        estimator.predict(val_matrix) for estimator in model.estimators_
    ]).astype(np.float32)


def pair_prediction_matrix(train_features, val_features, target):
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])

    return np.column_stack(neighbor_targets).astype(np.float32)


def combine_cached_predictions(cache, tree_q, pair_q, pair_weight, threshold):
    tree_pred = np.quantile(cache["tree_matrix"], tree_q, axis=1)
    pair_pred = np.quantile(cache["pair_matrix"], pair_q, axis=1)

    pred = np.clip(
        np.round((1.0 - pair_weight) * tree_pred + pair_weight * pair_pred, ROUND_DECIMALS),
        0.0,
        1.0,
    )

    override_mask = cache["nearest_dist"] < threshold
    pred[override_mask] = np.clip(
        np.round(cache["nearest_target"][override_mask], ROUND_DECIMALS),
        0.0,
        1.0,
    )
    return pred


In [4]:
train_raw = pd.read_csv(DATA_DIR / "train.csv")
y = train_raw[TARGET].reset_index(drop=True)
raw_input = train_raw.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
numeric_cols = raw_input.select_dtypes(include=["number"]).columns.tolist()

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)
fold_cache = []

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_input), start=1):
    train_raw_fold = raw_input.iloc[train_idx].reset_index(drop=True)
    val_raw_fold = raw_input.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)

    bounds = compute_iqr_bounds(train_raw_fold, WINSORIZE_COLUMNS)
    train_winsorized = apply_winsorize(train_raw_fold, bounds)
    val_winsorized = apply_winsorize(val_raw_fold, bounds)

    train_feat = add_features(train_winsorized)
    val_feat = add_features(val_winsorized)

    tree_matrix = tree_prediction_matrix(train_feat, val_feat, y_train)
    pair_matrix = pair_prediction_matrix(train_feat, val_feat, y_train)
    nearest_dist, nearest_target = nearest_neighbor_lookup(
        train_raw_fold, val_raw_fold, y_train, numeric_cols
    )

    fold_cache.append({
        "fold": fold,
        "val_idx": val_idx,
        "tree_matrix": tree_matrix,
        "pair_matrix": pair_matrix,
        "nearest_dist": nearest_dist,
        "nearest_target": nearest_target,
    })
    print(f"Fold {fold} 캐시 완료")


Fold 1 캐시 완료
Fold 2 캐시 완료
Fold 3 캐시 완료
Fold 4 캐시 완료
Fold 5 캐시 완료


## 2. 원본 설정 재현 확인

여기서 출력되는 기준선은 업로드한 원본 노트북의 설정과 동일해야 합니다.

In [5]:
def evaluate_setting(tree_q, pair_q, pair_weight, threshold):
    oof = np.zeros(len(y), dtype=float)
    for cache in fold_cache:
        oof[cache["val_idx"]] = combine_cached_predictions(
            cache, tree_q, pair_q, pair_weight, threshold
        )
    return mean_absolute_error(y, oof), oof


baseline_mae, baseline_oof = evaluate_setting(
    BASE_TREE_QUANTILE,
    BASE_PAIR_QUANTILE,
    BASE_PAIR_WEIGHT,
    BASE_OVERRIDE_THRESHOLD,
)

print(f"원본 설정 CV MAE: {baseline_mae:.8f}")
print("원본 설정:", {
    "tree_q": BASE_TREE_QUANTILE,
    "pair_q": BASE_PAIR_QUANTILE,
    "pair_weight": BASE_PAIR_WEIGHT,
    "threshold": BASE_OVERRIDE_THRESHOLD,
})


원본 설정 CV MAE: 0.14736334
원본 설정: {'tree_q': 0.54, 'pair_q': 0.48, 'pair_weight': 0.28, 'threshold': 0.2}


## 3. 1차 미세 탐색

원본 주변의 좁은 범위만 탐색합니다. 반올림과 모델 구조는 고정입니다.

In [6]:
TREE_Q_CANDIDATES = np.round(np.arange(0.50, 0.581, 0.01), 2)
PAIR_Q_CANDIDATES = np.round(np.arange(0.44, 0.521, 0.01), 2)
PAIR_WEIGHT_CANDIDATES = np.round(np.arange(0.20, 0.321, 0.01), 2)
THRESHOLD_CANDIDATES = np.round(np.arange(0.15, 0.251, 0.01), 2)

# 각 Fold와 분위수별 예측을 미리 계산해 반복 비용 절감
for cache in fold_cache:
    cache["tree_by_q"] = {
        float(q): np.quantile(cache["tree_matrix"], q, axis=1)
        for q in TREE_Q_CANDIDATES
    }
    cache["pair_by_q"] = {
        float(q): np.quantile(cache["pair_matrix"], q, axis=1)
        for q in PAIR_Q_CANDIDATES
    }

search_rows = []
total = (
    len(TREE_Q_CANDIDATES) * len(PAIR_Q_CANDIDATES)
    * len(PAIR_WEIGHT_CANDIDATES) * len(THRESHOLD_CANDIDATES)
)
count = 0

for tree_q in TREE_Q_CANDIDATES:
    for pair_q in PAIR_Q_CANDIDATES:
        for weight in PAIR_WEIGHT_CANDIDATES:
            for threshold in THRESHOLD_CANDIDATES:
                oof = np.zeros(len(y), dtype=float)

                for cache in fold_cache:
                    tree_pred = cache["tree_by_q"][float(tree_q)]
                    pair_pred = cache["pair_by_q"][float(pair_q)]
                    pred = np.clip(
                        np.round((1.0 - weight) * tree_pred + weight * pair_pred, 2),
                        0.0,
                        1.0,
                    )
                    mask = cache["nearest_dist"] < threshold
                    pred[mask] = np.clip(
                        np.round(cache["nearest_target"][mask], 2), 0.0, 1.0
                    )
                    oof[cache["val_idx"]] = pred

                mae = mean_absolute_error(y, oof)
                search_rows.append({
                    "tree_quantile": float(tree_q),
                    "pair_quantile": float(pair_q),
                    "pair_weight": float(weight),
                    "override_threshold": float(threshold),
                    "cv_mae": mae,
                })

                count += 1
                if count % 1000 == 0 or count == total:
                    print(f"탐색 {count:,}/{total:,} 완료")

search_df = (
    pd.DataFrame(search_rows)
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

display(search_df.head(30))
print(f"기준선: {baseline_mae:.8f}")
print(f"탐색 최적: {search_df.loc[0, 'cv_mae']:.8f}")
print(f"CV 개선폭: {baseline_mae - search_df.loc[0, 'cv_mae']:+.8f}")


탐색 1,000/11,583 완료
탐색 2,000/11,583 완료
탐색 3,000/11,583 완료
탐색 4,000/11,583 완료
탐색 5,000/11,583 완료
탐색 6,000/11,583 완료
탐색 7,000/11,583 완료
탐색 8,000/11,583 완료
탐색 9,000/11,583 완료
탐색 10,000/11,583 완료
탐색 11,000/11,583 완료
탐색 11,583/11,583 완료


,tree_quantile,pair_quantile,pair_weight,override_threshold,cv_mae
0,0.52,0.48,0.20,0.18,0.147093
1,0.52,0.48,0.20,0.15,0.147093
2,0.52,0.48,0.20,0.16,0.147093
3,0.52,0.48,0.20,0.17,0.147093
4,0.52,0.48,0.20,0.19,0.147093
5,0.52,0.48,0.20,0.20,0.147093
6,0.52,0.48,0.20,0.21,0.147093
7,0.52,0.48,0.20,0.22,0.147093
8,0.52,0.48,0.20,0.23,0.147093
9,0.52,0.48,0.20,0.24,0.147093


기준선: 0.14736334
탐색 최적: 0.14709333
CV 개선폭: +0.00027000


## 4. 안정적인 제출 후보 선택

CV 최상위 조합이 서로 거의 같을 수 있으므로, 설정이 겹치지 않는 후보 3개를 선택합니다. 첫 번째가 CV 최적이고, 나머지는 분위수·가중치·임계값이 조금 다른 상위 조합입니다.

In [7]:
def choose_diverse_candidates(results, n=3):
    selected = []
    for _, row in results.iterrows():
        candidate = row.to_dict()
        if not selected:
            selected.append(candidate)
        else:
            sufficiently_different = all(
                abs(candidate["tree_quantile"] - old["tree_quantile"]) >= 0.01
                or abs(candidate["pair_quantile"] - old["pair_quantile"]) >= 0.01
                or abs(candidate["pair_weight"] - old["pair_weight"]) >= 0.02
                or abs(candidate["override_threshold"] - old["override_threshold"]) >= 0.02
                for old in selected
            )
            if sufficiently_different:
                selected.append(candidate)
        if len(selected) >= n:
            break
    return pd.DataFrame(selected)

submission_candidates = choose_diverse_candidates(search_df, n=3)
display(submission_candidates)


,tree_quantile,pair_quantile,pair_weight,override_threshold,cv_mae
0,0.52,0.48,0.2,0.18,0.147093
1,0.52,0.48,0.2,0.15,0.147093
2,0.52,0.48,0.2,0.20,0.147093


## 5. 최종 학습 및 제출 파일 3개 생성

모델은 한 번만 학습하고, 후보별 후처리만 다르게 적용합니다.

In [9]:
test_raw = pd.read_csv(DATA_DIR / "test.csv")
submission_template = pd.read_csv(DATA_DIR / "sample_submission.csv")
test_input = test_raw.drop(columns=[ID_COLUMN]).reset_index(drop=True)

# 전체 학습 데이터 기준 전처리
bounds = compute_iqr_bounds(raw_input, WINSORIZE_COLUMNS)
train_winsorized = apply_winsorize(raw_input, bounds)
test_winsorized = apply_winsorize(test_input, bounds)
train_feat = add_features(train_winsorized)
test_feat = add_features(test_winsorized)

print("전체 데이터 ExtraTrees 학습 시작")
final_tree_matrix = tree_prediction_matrix(train_feat, test_feat, y)
print("Pair 예측 캐시 생성")
final_pair_matrix = pair_prediction_matrix(train_feat, test_feat, y)
final_nearest_dist, final_nearest_target = nearest_neighbor_lookup(
    raw_input, test_input, y, numeric_cols
)

saved_paths = []
for rank, row in submission_candidates.iterrows():
    tree_q = float(row["tree_quantile"])
    pair_q = float(row["pair_quantile"])
    weight = float(row["pair_weight"])
    threshold = float(row["override_threshold"])

    tree_pred = np.quantile(final_tree_matrix, tree_q, axis=1)
    pair_pred = np.quantile(final_pair_matrix, pair_q, axis=1)
    final_pred = np.clip(
        np.round((1.0 - weight) * tree_pred + weight * pair_pred, 2),
        0.0,
        1.0,
    )
    mask = final_nearest_dist < threshold
    final_pred[mask] = np.clip(
        np.round(final_nearest_target[mask], 2), 0.0, 1.0
    )

    output = submission_template.copy()
    output[TARGET] = final_pred

    filename = (
        f"submit_tuned_rank{rank+1}"
        f"_tq{tree_q:.2f}_pq{pair_q:.2f}"
        f"_w{weight:.2f}_th{threshold:.2f}.csv"
    )
    path = OUTPUT_DIR / filename
    output.to_csv(path, index=False, encoding="utf-8")
    saved_paths.append(path)

    print(f"Saved rank {rank+1}: {path}")
    print(f"  CV MAE={row['cv_mae']:.8f}, override={mask.sum()} rows")

print("생성 완료")
for path in saved_paths:
    print(path)


전체 데이터 ExtraTrees 학습 시작
Pair 예측 캐시 생성
Saved rank 1: /Users/bitsaem/Desktop/stress_project/8_6/submit_tuned_rank1_tq0.52_pq0.48_w0.20_th0.18.csv
  CV MAE=0.14709333, override=634 rows
Saved rank 2: /Users/bitsaem/Desktop/stress_project/8_6/submit_tuned_rank2_tq0.52_pq0.48_w0.20_th0.15.csv
  CV MAE=0.14709333, override=604 rows
Saved rank 3: /Users/bitsaem/Desktop/stress_project/8_6/submit_tuned_rank3_tq0.52_pq0.48_w0.20_th0.20.csv
  CV MAE=0.14709333, override=646 rows
생성 완료
/Users/bitsaem/Desktop/stress_project/8_6/submit_tuned_rank1_tq0.52_pq0.48_w0.20_th0.18.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_tuned_rank2_tq0.52_pq0.48_w0.20_th0.15.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_tuned_rank3_tq0.52_pq0.48_w0.20_th0.20.csv
